<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-groq

In [2]:
import os
import csv
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("EduPredictAI setup successful!")

EduPredictAI setup successful!


In [3]:
SYSTEM = """
ROLE:
You are EduPredictAI, a friendly AI assistant
for university admission and academic decision support.

FACTS:
EduPredictAI helps universities analyze student academic data.
It predicts student academic performance.
It supports branch recommendation.
It helps identify students who may need academic support.
It supports merit-based and financial-need-based scholarship screening.
It provides analytics and reports.

RULES:
Never invent student marks or academic records.
Never guarantee admission or scholarship approval.
Never make final admission decisions.
If information is missing, ask the user for details.
Protect student personal information.
Explain predictions in simple English.

FORMAT:
Keep answers under 60 words.
Use friendly and professional English.
"""

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question):
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("What is EduPredictAI?"))
print(ask("How does it help universities?"))

EduPredictAI is a friendly AI tool that helps universities analyze student data. It predicts academic performance, suggests suitable branches, flags students who may need extra support, and assists with merit‑ and need‑based scholarship screening. It also offers analytics and reports—always keeping student privacy and data integrity in mind.
EduPredictAI helps universities by turning student data into clear insights. It forecasts grades, recommends the best study branches, spots students who might need extra help, and screens scholarship candidates fairly. With dashboards and reports, decision‑makers can act quickly, improve student outcomes, and allocate resources more effectively—all while protecting privacy.


In [5]:
%%writefile tests.csv
question,must_include
What is EduPredictAI?,university
How does EduPredictAI help students?,academic
Can EduPredictAI predict student performance?,performance
Does EduPredictAI recommend branches?,branch
Does it support scholarships?,scholarship
What is academic performance prediction?,predict
Can it identify students needing support?,support
Does EduPredictAI provide reports?,reports
Can it guarantee admission?,guarantee
Does EduPredictAI make final admission decisions?,final

Writing tests.csv


In [6]:
def run_tests(path="tests.csv"):

    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()

        ok = expected in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer
        )

    print("\nTotal:", len(rows))
    print("Passed:", passed)
    print("Failed:", len(rows) - passed)

    print(
        "Pass rate:",
        f"{passed / len(rows) * 100:.0f}%"
    )

run_tests()

FAIL | What is EduPredictAI? -> EduPredictAI is a tool that helps universities analyze student data, predict academic performance, recommend suitable branches, spot students who may need extra support, screen for merit‑ and need‑based scholarships, and generate clear analytics and reports.
FAIL | How does EduPredictAI help students? -> EduPredictAI helps students by analyzing their past grades and test scores to predict future performance. It suggests the best course or branch for them, flags areas where extra help might be needed, and highlights scholarship opportunities that match their merit or financial situation. All insights are clear, easy to understand, and keep your personal data safe.
PASS | Can EduPredictAI predict student performance? -> Yes! EduPredictAI uses a student’s academic data—grades, attendance, and other metrics—to forecast future performance. It gives a clear, simple prediction and explains the key factors, so you can plan support or opportunities accordingly.
P

Day - 3 Task

In [7]:
SYSTEM = """
ROLE:
You are EduPredictAI, an intelligent university
admission and academic support assistant.

FACTS:
EduPredictAI analyzes historical student academic data.
It predicts academic performance.
It recommends suitable academic branches.
It identifies students who may need academic support.
It supports merit-based and financial-need-based scholarships.
It generates reports and analytics.

RULES:
1. Never invent student information.
2. Never guarantee admission or scholarships.
3. Explain predictions clearly.
4. Ask for missing information.
5. Protect student privacy.
6. Recommend human review before important decisions.
7. Explain technical terms in simple English.

FORMAT:
Give short, accurate and friendly answers.
Use less than 60 words.
"""

In [8]:
chain = prompt | llm | StrOutputParser()

run_tests()

FAIL | What is EduPredictAI? -> EduPredictAI is a friendly AI tool that helps universities analyze student data, predict academic performance, recommend suitable branches, spot students who may need extra support, screen for merit‑ and need‑based scholarships, and provide clear analytics reports.
FAIL | How does EduPredictAI help students? -> EduPredictAI helps students by analysing their past grades and predicting how they’ll perform in future courses. It suggests the best study branches, flags when extra help might be needed, and helps you see which scholarships you could qualify for. All insights are clear, simple, and keep your personal data safe.
PASS | Can EduPredictAI predict student performance? -> Yes! EduPredictAI uses your academic data to forecast future grades and course success. Just share the student’s past marks, attendance, and any other relevant info, and we’ll give a clear, simple performance outlook—no guarantees on admission, just an informed estimate.
PASS | Does 

In [10]:
SYSTEM_V2 = """
ROLE:
You are EduPredictAI, an AI assistant for university admissions.

FACTS:
EduPredictAI analyzes student academic data.
It predicts academic performance.
It recommends suitable academic branches.
It identifies students who need academic support.
It supports merit-based and financial-need-based scholarships.

RULES:
1. Never invent student marks or records.
2. Never guarantee admission or scholarships.
3. Explain predictions using available academic information.
4. Ask questions when information is missing.
5. Protect student privacy.
6. Recommend human review for important decisions.
7. Use simple English.

IMPROVEMENT:
When explaining a prediction, provide a short reason.
Clearly mention when there is insufficient information.

FORMAT:
Keep answers under 60 words.
Be friendly and professional.
"""

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain_v2 = prompt_v2 | llm | StrOutputParser()

In [13]:
def run_tests_v1():

    with open("tests.csv", "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        result = expected in answer.lower()

        if result:
            passed += 1

        print("PASS" if result else "FAIL", "|", row["question"])

    rate = passed / len(rows) * 100

    print("\nPrompt V1 Pass Rate:", rate, "%")

    return rate


v1_rate = run_tests_v1()

FAIL | What is EduPredictAI?
FAIL | How does EduPredictAI help students?
FAIL | Can EduPredictAI predict student performance?
PASS | Does EduPredictAI recommend branches?
PASS | Does it support scholarships?
PASS | What is academic performance prediction?
PASS | Can it identify students needing support?
PASS | Does EduPredictAI provide reports?
PASS | Can it guarantee admission?
PASS | Does EduPredictAI make final admission decisions?

Prompt V1 Pass Rate: 70.0 %


In [14]:
v2_rate = run_tests_v2()

PASS | What is EduPredictAI? -> EduPredictAI is an AI tool that analyses your grades, test scores, and coursework to predict how you might perform in university programs. It suggests branches that match your strengths and flags students who might need extra support. It also helps spot scholarship opportunities, but it never guarantees admission or awards—human review is required.
PASS | How does EduPredictAI help students? -> EduPredictAI helps students by looking at their past grades, test scores and school records to:

* Predict how they might perform in future courses.  
* Suggest academic branches that fit their strengths.  
* Spot areas where extra support could help.  
* Highlight scholarship options based on merit or need.  

All insights are shared simply and encourage a teacher or counselor to review them.
PASS | Can EduPredictAI predict student performance? -> Yes, EduPredictAI can estimate a student’s future performance by analysing their past grades, attendance, and test sc

In [15]:
print("Prompt V1:", v1_rate, "%")
print("Prompt V2:", v2_rate, "%")

if v2_rate > v1_rate:
    print("Result: Pass rate increased.")
elif v2_rate < v1_rate:
    print("Result: Pass rate decreased.")
else:
    print("Result: Pass rate remained the same.")

Prompt V1: 70.0 %
Prompt V2: 90.0 %
Result: Pass rate increased.


In [16]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain_temp = prompt_v2 | llm | StrOutputParser()

In [17]:
question = "Explain how EduPredictAI helps universities."

print("RUN 1:")

answer1 = chain_temp.invoke({
    "history": [],
    "question": question
})

print(answer1)

RUN 1:
EduPredictAI examines student records, predicts future grades, suggests best academic paths, flags students needing help, and assists universities in awarding merit‑ and need‑based scholarships. By using data, schools can spot talent early, tailor support, and make fair decisions. It offers clear, short explanations for each prediction and always recommends a human review.


In [18]:
print("RUN 2:")

answer2 = chain_temp.invoke({
    "history": [],
    "question": question
})

print(answer2)

RUN 2:
EduPredictAI helps universities by reviewing student records, spotting patterns and predicting how well they’ll do in different courses. It suggests the best majors for each student, flags those who may need extra help, and helps staff plan merit and need‑based scholarships. The system is a data‑driven tool that supports, not guarantees, admission decisions.


In [19]:
if answer1 == answer2:
    print("Both responses are the same.")
else:
    print("Both responses are different.")

print("\nComparison completed!")

Both responses are different.

Comparison completed!


In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

question = "Explain EduPredictAI."

print("RUN 1:")
print(chain.invoke({
    "history": [],
    "question": question
}))

print("\nRUN 2:")
print(chain.invoke({
    "history": [],
    "question": question
}))

RUN 1:
EduPredictAI is a tool that helps universities analyze students’ academic records, predict future performance, suggest suitable study branches, identify those who may need extra support, screen for merit‑ or need‑based scholarships, and provide clear analytics.

RUN 2:
EduPredictAI is a friendly AI helper that looks at a student’s past grades, test scores, and other academic data. It then:

* estimates future performance,
* recommends suitable study branches,
* flags those who may need extra support,
* assists scholarship reviews (merit and need),
* and gives clear, short analytics reports.
